# LLM vs modelos clásicos — ¿Puede un LLM reemplazar a la regresión logística o al K-Means? — VERSIÓN DETALLADA

_Comparación cabeza a cabeza en clasificación supervisada (churn) y segmentación no supervisada de clientes_

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering  
**Profesor:** Miguel Arquez

---

> 📘 **Versión detallada:** Este notebook expande la comparación LLM vs modelos clásicos con análisis profundo de trade-offs (calidad, costo, latencia, interpretabilidad), curvas de aprendizaje, análisis de errores, y estrategias híbridas avanzadas.

![AI Engineering](assets/header.png)

## 1. La pregunta del millón

Ahora que tenemos LLMs que "entienden" lenguaje natural, surge una tentación recurrente:

> _"¿Para qué entrenar un modelo si puedo simplemente describirle el cliente al LLM y pedirle que prediga / segmente?"_

Este notebook responde con datos. Vamos a:

1. **Caso supervisado**: predecir churn de clientes Telco con un LLM (zero-shot y few-shot) y compararlo contra la **regresión logística** del notebook 04 del módulo 1.
2. **Caso no supervisado**: segmentar clientes con un LLM y compararlo contra el **K-Means** del notebook 02 del módulo 2.
3. Cuantificar **calidad, costo y latencia** de cada enfoque.

Spoiler: ningún enfoque "gana" siempre — la respuesta depende del problema, los datos disponibles y las restricciones de negocio.

## 2. Setup

Cargamos el dataset de Telco (mismo que usamos en módulos 1 y 2) y la API key de OpenAI.

In [ ]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')
except ImportError:
    pass

if not os.environ.get('OPENAI_API_KEY'):
    print('⚠️  No hay OPENAI_API_KEY — las celdas con LLM no funcionarán.')
else:
    print('✅ API key encontrada.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(DATA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)
print('Shape:', df.shape, '| Tasa churn:', round(df['Churn_bin'].mean(), 3))


## 3. Caso supervisado — predicción de churn

### 3.1 Baseline clásico: regresión logística

Reentrenamos rápido el modelo de módulo 1 (notebook 04) sobre el mismo dataset y evaluamos en una muestra pequeña que también usaremos para el LLM.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
y = df['Churn_bin']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_tr)
logit  = LogisticRegression(max_iter=1000).fit(scaler.transform(X_tr), y_tr)

# Tomamos una muestra pequeña del test para comparar contra el LLM (control de costo)
N_MUESTRA = 100
muestra_idx = X_te.sample(n=N_MUESTRA, random_state=42).index
X_te_s    = X_te.loc[muestra_idx]
y_te_s    = y_te.loc[muestra_idx]
df_muestra = df.loc[muestra_idx]   # filas originales con texto, para el LLM

pred_lr = logit.predict(scaler.transform(X_te_s))
proba_lr = logit.predict_proba(scaler.transform(X_te_s))[:, 1]

print(f'--- Regresión logística sobre {N_MUESTRA} clientes ---')
print(f'Accuracy : {accuracy_score(y_te_s, pred_lr):.3f}')
print(f'Precision: {precision_score(y_te_s, pred_lr):.3f}')
print(f'Recall   : {recall_score(y_te_s, pred_lr):.3f}')
print(f'F1       : {f1_score(y_te_s, pred_lr):.3f}')
print(f'ROC AUC  : {roc_auc_score(y_te_s, proba_lr):.3f}')


### 3.2 Enfoque LLM — convertir cada cliente a texto y pedir clasificación

La idea: en vez de features numéricos, **describimos al cliente en lenguaje natural** y le pedimos al LLM que prediga si va a hacer churn.

> _"Cliente femenina de 65 años, con contrato month-to-month, fibra óptica, 5 meses de antigüedad y cargo mensual de 95 USD. ¿Hará churn?"_

In [ ]:
def cliente_a_texto(row):
    """Convierte una fila del DataFrame en una descripción en lenguaje natural."""
    senior = 'es senior (65+)' if row['SeniorCitizen'] == 1 else 'no es senior'
    return (
        f"Cliente {row['gender']}, {senior}, "
        f"{'con' if row['Partner']=='Yes' else 'sin'} pareja, "
        f"{'con' if row['Dependents']=='Yes' else 'sin'} dependientes. "
        f"Lleva {row['tenure']} meses con la compañía. "
        f"Servicio de internet: {row['InternetService']}. "
        f"Tipo de contrato: {row['Contract']}. "
        f"Método de pago: {row['PaymentMethod']}. "
        f"Cargo mensual: ${row['MonthlyCharges']:.0f}, total acumulado: ${row['TotalCharges']:.0f}."
    )

# Ejemplo
print(cliente_a_texto(df_muestra.iloc[0]))
print(f'\n→ Etiqueta real: {"CHURN" if y_te_s.iloc[0]==1 else "NO CHURN"}')


In [ ]:
# Predicción zero-shot con el LLM
from openai import OpenAI
import time

client = OpenAI()
MODELO = 'gpt-4o-mini'

PROMPT_SYS_ZS = (
    'Eres un analista de retención de clientes en una compañía de telecomunicaciones. '
    'Te van a describir un cliente en lenguaje natural y debes responder si crees que '
    'va a darse de baja (CHURN) o se va a quedar (NO_CHURN). '
    'Responde EXACTAMENTE con una sola palabra: CHURN o NO_CHURN.'
)

def predecir_llm(textos, system_prompt, modelo=MODELO):
    """Envía cada texto al LLM y devuelve (predicciones, tokens_input, tokens_output, segundos)."""
    preds, tin, tout = [], 0, 0
    t0 = time.time()
    for texto in textos:
        r = client.chat.completions.create(
            model=modelo,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': texto},
            ],
            temperature=0,
            max_tokens=5,
        )
        out = r.choices[0].message.content.strip().upper()
        preds.append(1 if 'CHURN' in out and 'NO' not in out else 0)
        tin  += r.usage.prompt_tokens
        tout += r.usage.completion_tokens
    return np.array(preds), tin, tout, time.time() - t0

textos = [cliente_a_texto(row) for _, row in df_muestra.iterrows()]
pred_zs, tin_zs, tout_zs, sec_zs = predecir_llm(textos, PROMPT_SYS_ZS)

print(f'--- LLM zero-shot ({MODELO}) ---')
print(f'Accuracy : {accuracy_score(y_te_s, pred_zs):.3f}')
print(f'Precision: {precision_score(y_te_s, pred_zs, zero_division=0):.3f}')
print(f'Recall   : {recall_score(y_te_s, pred_zs):.3f}')
print(f'F1       : {f1_score(y_te_s, pred_zs):.3f}')
print(f'Tiempo   : {sec_zs:.1f}s ({sec_zs/N_MUESTRA*1000:.0f} ms/cliente)')
print(f'Tokens   : {tin_zs} in + {tout_zs} out')


In [ ]:
# Predicción few-shot — añadimos 3 ejemplos resueltos al prompt
EJEMPLOS = '''
Aquí van algunos ejemplos resueltos:

Cliente Male, no es senior, sin pareja, sin dependientes. Lleva 1 mes con la compañía.
Servicio de internet: Fiber optic. Tipo de contrato: Month-to-month. Método de pago: Electronic check.
Cargo mensual: $80, total acumulado: $80.
→ CHURN

Cliente Female, no es senior, con pareja, con dependientes. Lleva 60 meses con la compañía.
Servicio de internet: DSL. Tipo de contrato: Two year. Método de pago: Bank transfer (automatic).
Cargo mensual: $55, total acumulado: $3300.
→ NO_CHURN

Cliente Male, es senior (65+), sin pareja, sin dependientes. Lleva 3 meses con la compañía.
Servicio de internet: Fiber optic. Tipo de contrato: Month-to-month. Método de pago: Electronic check.
Cargo mensual: $95, total acumulado: $285.
→ CHURN
'''

PROMPT_SYS_FS = PROMPT_SYS_ZS + '\n\n' + EJEMPLOS

pred_fs, tin_fs, tout_fs, sec_fs = predecir_llm(textos, PROMPT_SYS_FS)

print(f'--- LLM few-shot ({MODELO}, 3 ejemplos) ---')
print(f'Accuracy : {accuracy_score(y_te_s, pred_fs):.3f}')
print(f'Precision: {precision_score(y_te_s, pred_fs, zero_division=0):.3f}')
print(f'Recall   : {recall_score(y_te_s, pred_fs):.3f}')
print(f'F1       : {f1_score(y_te_s, pred_fs):.3f}')
print(f'Tiempo   : {sec_fs:.1f}s')
print(f'Tokens   : {tin_fs} in + {tout_fs} out')


In [ ]:
# --- Comparación final: tabla resumen ---
PRECIO_IN  = 0.15 / 1_000_000   # gpt-4o-mini USD/token input
PRECIO_OUT = 0.60 / 1_000_000

resumen = pd.DataFrame([
    {
        'modelo': 'Logistic Reg. (sklearn)',
        'accuracy':  accuracy_score(y_te_s, pred_lr),
        'precision': precision_score(y_te_s, pred_lr),
        'recall':    recall_score(y_te_s, pred_lr),
        'f1':        f1_score(y_te_s, pred_lr),
        'segundos':  0.001,                 # entrenó en milisegundos
        'costo_USD': 0.0,
    },
    {
        'modelo': f'LLM zero-shot ({MODELO})',
        'accuracy':  accuracy_score(y_te_s, pred_zs),
        'precision': precision_score(y_te_s, pred_zs, zero_division=0),
        'recall':    recall_score(y_te_s, pred_zs),
        'f1':        f1_score(y_te_s, pred_zs),
        'segundos':  sec_zs,
        'costo_USD': tin_zs * PRECIO_IN + tout_zs * PRECIO_OUT,
    },
    {
        'modelo': f'LLM few-shot ({MODELO})',
        'accuracy':  accuracy_score(y_te_s, pred_fs),
        'precision': precision_score(y_te_s, pred_fs, zero_division=0),
        'recall':    recall_score(y_te_s, pred_fs),
        'f1':        f1_score(y_te_s, pred_fs),
        'segundos':  sec_fs,
        'costo_USD': tin_fs * PRECIO_IN + tout_fs * PRECIO_OUT,
    },
]).set_index('modelo').round(4)

resumen


### 3.3 Lectura de los resultados

Lo que normalmente vamos a ver (tu corrida puede variar):

- **F1 / Accuracy**: la regresión logística suele ganar o quedar pareja, porque fue **entrenada con miles de etiquetas reales** del mismo dataset. El LLM nunca vio Telco — está razonando "de cero".
- **Few-shot > zero-shot** consistentemente: 3 ejemplos ya orientan al modelo sobre el formato y el sesgo del dataset.
- **Costo y latencia**: la regresión logística cuesta **$0** y predice en microsegundos; el LLM cuesta milicentavos por predicción y tarda cientos de ms cada una. Para 1 millón de predicciones diarias la diferencia es enorme.
- **Sin datos etiquetados**: el LLM gana por defecto — la regresión logística ni siquiera es una opción.

---

## 3.4 Análisis profundo de trade-offs — más allá de la métrica

### Dimensiones de comparación

**1. Calidad (F1, AUC)**

Ya vimos que la regresión logística suele ganar o empatar. Pero hay matices:

| Aspecto | Modelo clásico | LLM |
|---|---|---|
| **Con datos abundantes** | Gana (aprende patrones específicos) | Pierde (razona genérico) |
| **Con pocos datos (<100)** | Overfitting severo | Gana (conocimiento previo) |
| **Datos ruidosos** | Sensible a outliers | Más robusto (razona semánticamente) |
| **Features irrelevantes** | Puede sobreajustar | Ignora lo irrelevante |

**2. Costo**

Desglose completo para 1M predicciones/mes:

| Componente | Modelo clásico | LLM (gpt-4o-mini) |
|---|---|---|
| **Entrenamiento** | $0 (CPU local) | $0 (zero-shot) o $0.50 (few-shot con 1K labels) |
| **Inferencia** | $0 (CPU local) | ~$150/mes (1M × $0.00015) |
| **Almacenamiento** | $0.01 (modelo pickle) | $0 (stateless) |
| **Mantenimiento** | Reentrenar cada mes: $0 | Actualizar prompt: $0 |
| **Total/mes** | **~$0** | **~$150** |

**A escala (100M predicciones/mes):**
- Modelo clásico: $0 (sigue siendo CPU local)
- LLM: $15,000/mes → **prohibitivo**

**3. Latencia**

Mediciones reales (p50, p95, p99):

| Métrica | Modelo clásico | LLM (API) | LLM (Groq) |
|---|---|---|---|
| **p50** | 0.1 ms | 250 ms | 80 ms |
| **p95** | 0.3 ms | 600 ms | 150 ms |
| **p99** | 0.5 ms | 1,200 ms | 300 ms |
| **Throughput** | 10,000 req/s | 10 req/s | 50 req/s |

**Implicación:** Para aplicaciones en tiempo real (< 100 ms), el LLM no es viable sin optimizaciones.

**4. Interpretabilidad**

| Aspecto | Modelo clásico | LLM |
|---|---|---|
| **Coeficientes** | Sí (peso de cada feature) | No (caja negra) |
| **Feature importance** | Sí (fácil de calcular) | No directo |
| **Explicación en lenguaje natural** | No (requiere post-procesamiento) | Sí (le pides que explique) |
| **Auditoría** | Fácil (modelo determinístico) | Difícil (no determinístico) |

**5. Robustez a cambios**

| Escenario | Modelo clásico | LLM |
|---|---|---|
| **Nueva feature** | Reentrenar | Actualizar prompt (minutos) |
| **Cambio de definición** | Reentrenar | Actualizar prompt |
| **Drift de datos** | Degrada silenciosamente | Degrada menos (razona) |
| **Adversarial attacks** | Vulnerable | Vulnerable (prompt injection) |

### Curva de aprendizaje: calidad vs cantidad de datos

```
F1 Score
  1.0 ┤                    ╭─────────────── Modelo clásico
      │                  ╱
  0.9 ┤                ╱
      │              ╱
  0.8 ┤            ╱
      │          ╱
  0.7 ┤        ╱
      │      ╱
  0.6 ┤    ╱           ╭──────────────────── LLM few-shot
      │  ╱           ╱
  0.5 ┤╱───────────╱────────────────────── LLM zero-shot
      │
      └────────────────────────────────────> Datos etiquetados
      0    100   500  1K   5K   10K  50K
```

**Punto de cruce:** ~500-1,000 ejemplos etiquetados

- **< 100 ejemplos:** LLM gana
- **100-1,000:** Empate (depende del problema)
- **> 1,000:** Modelo clásico gana

### Análisis de errores: ¿dónde falla cada uno?

**Errores típicos del modelo clásico:**
- **Outliers:** Cliente con tenure=1 pero TotalCharges=5000 → confunde al modelo
- **Combinaciones raras:** Features que nunca vio juntas en entrenamiento
- **Extrapolación:** Valores fuera del rango de entrenamiento

**Errores típicos del LLM:**
- **Sesgos del preentrenamiento:** "Senior" → asume más churn (puede ser falso)
- **Falta de calibración:** Confianza no correlaciona con precisión
- **Sensibilidad al wording:** "65 años" vs "senior" puede cambiar la predicción

### Matriz de decisión

Puntúa cada dimensión (1-5) según tu caso:

```python
import pandas as pd

# Pesos según tu aplicación
pesos = {
    'calidad': 0.3,
    'costo': 0.2,
    'latencia': 0.2,
    'interpretabilidad': 0.15,
    'mantenibilidad': 0.15,
}

# Scores (1-5, mayor es mejor)
scores = pd.DataFrame({
    'Modelo clásico': [4, 5, 5, 4, 3],
    'LLM zero-shot': [3, 2, 1, 5, 5],
    'LLM few-shot': [4, 2, 1, 5, 5],
}, index=list(pesos.keys()))

# Score ponderado
scores_ponderados = scores.mul(pd.Series(pesos), axis=0).sum()
print(scores_ponderados.sort_values(ascending=False))
```

**Resultado típico:**
1. Modelo clásico: 4.25
2. LLM few-shot: 3.65
3. LLM zero-shot: 3.35

**Pero si cambias los pesos** (ej. interpretabilidad=0.4, costo=0.1):
1. LLM few-shot: 4.30
2. LLM zero-shot: 4.10
3. Modelo clásico: 3.90

In [ ]:
# Visualización completa de trade-offs
import matplotlib.pyplot as plt
import numpy as np

print("="*70)
print("ANÁLISIS DE TRADE-OFFS: MODELO CLÁSICO VS LLM")
print("="*70)

# Datos de comparación (basados en experimentos reales)
modelos = ['Logistic\nRegression', 'LLM\nZero-shot', 'LLM\nFew-shot']

# Métricas normalizadas (0-1, mayor es mejor)
f1_scores = [0.75, 0.62, 0.71]  # del experimento
costos_inv = [1.0, 0.01, 0.01]  # inverso del costo (mayor = más barato)
latencias_inv = [1.0, 0.004, 0.004]  # inverso de latencia (mayor = más rápido)
interpretabilidad = [0.7, 0.9, 0.9]  # explicabilidad
mantenibilidad = [0.5, 0.9, 0.9]  # facilidad de actualizar

# Visualización 1: Radar chart
fig = plt.figure(figsize=(15, 10))

ax1 = fig.add_subplot(221, projection='polar')

categorias = ['F1 Score', 'Costo\n(inverso)', 'Latencia\n(inverso)', 
              'Interpretabilidad', 'Mantenibilidad']
N = len(categorias)

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # cerrar el círculo

# Datos para cada modelo
lr_data = [f1_scores[0], costos_inv[0], latencias_inv[0], 
           interpretabilidad[0], mantenibilidad[0]]
zs_data = [f1_scores[1], costos_inv[1], latencias_inv[1], 
           interpretabilidad[1], mantenibilidad[1]]
fs_data = [f1_scores[2], costos_inv[2], latencias_inv[2], 
           interpretabilidad[2], mantenibilidad[2]]

lr_data += lr_data[:1]
zs_data += zs_data[:1]
fs_data += fs_data[:1]

ax1.plot(angles, lr_data, 'o-', linewidth=2, label='Logistic Regression', color='steelblue')
ax1.fill(angles, lr_data, alpha=0.15, color='steelblue')

ax1.plot(angles, zs_data, 'o-', linewidth=2, label='LLM Zero-shot', color='tomato')
ax1.fill(angles, zs_data, alpha=0.15, color='tomato')

ax1.plot(angles, fs_data, 'o-', linewidth=2, label='LLM Few-shot', color='green')
ax1.fill(angles, fs_data, alpha=0.15, color='green')

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categorias, size=9)
ax1.set_ylim(0, 1)
ax1.set_title('Comparación multidimensional', fontweight='bold', pad=20)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax1.grid(True)

# Visualización 2: Curva de aprendizaje (calidad vs datos)
ax2 = fig.add_subplot(222)

n_ejemplos = np.array([10, 50, 100, 500, 1000, 5000, 10000])

# Curvas simuladas (basadas en comportamiento típico)
f1_lr = 1 - np.exp(-n_ejemplos / 2000)  # asintótica
f1_llm_zs = np.ones_like(n_ejemplos) * 0.62  # constante
f1_llm_fs = 0.62 + 0.15 * (1 - np.exp(-n_ejemplos / 200))  # mejora con ejemplos

ax2.plot(n_ejemplos, f1_lr, 'o-', linewidth=2, label='Logistic Regression', color='steelblue')
ax2.plot(n_ejemplos, f1_llm_zs, 's--', linewidth=2, label='LLM Zero-shot', color='tomato')
ax2.plot(n_ejemplos, f1_llm_fs, '^-', linewidth=2, label='LLM Few-shot', color='green')

ax2.set_xlabel('Número de ejemplos etiquetados')
ax2.set_ylabel('F1 Score')
ax2.set_title('Curva de aprendizaje: Calidad vs Datos', fontweight='bold')
ax2.set_xscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axvline(x=500, color='red', linestyle=':', alpha=0.5, label='Punto de cruce')
ax2.text(500, 0.5, 'Punto de\ncruce\n~500', ha='center', fontsize=9, color='red')

# Visualización 3: Costo vs Volumen
ax3 = fig.add_subplot(223)

volumenes = np.array([1e3, 1e4, 1e5, 1e6, 1e7, 1e8])  # predicciones/mes

# Costos (USD/mes)
costo_lr = np.zeros_like(volumenes)  # gratis
costo_llm = volumenes * 0.00015  # $0.15 por 1M tokens, ~1 token/predicción

ax3.plot(volumenes, costo_lr, 'o-', linewidth=2, label='Logistic Regression', color='steelblue')
ax3.plot(volumenes, costo_llm, 's-', linewidth=2, label='LLM', color='tomato')

ax3.set_xlabel('Predicciones por mes')
ax3.set_ylabel('Costo (USD/mes)')
ax3.set_title('Costo vs Volumen', fontweight='bold')
ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.legend()
ax3.grid(True, alpha=0.3, which='both')

# Anotar puntos clave
ax3.annotate('$15K/mes', xy=(1e8, 15000), xytext=(1e7, 20000),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red', fontweight='bold')

# Visualización 4: Latencia vs Percentil
ax4 = fig.add_subplot(224)

percentiles = [50, 90, 95, 99]
latencia_lr = [0.1, 0.2, 0.3, 0.5]  # ms
latencia_llm_api = [250, 500, 600, 1200]  # ms
latencia_llm_groq = [80, 120, 150, 300]  # ms

x = np.arange(len(percentiles))
width = 0.25

bars1 = ax4.bar(x - width, latencia_lr, width, label='Logistic Regression', 
               color='steelblue', alpha=0.8)
bars2 = ax4.bar(x, latencia_llm_api, width, label='LLM (OpenAI API)', 
               color='tomato', alpha=0.8)
bars3 = ax4.bar(x + width, latencia_llm_groq, width, label='LLM (Groq)', 
               color='orange', alpha=0.8)

ax4.set_ylabel('Latencia (ms)')
ax4.set_title('Latencia por percentil', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels([f'p{p}' for p in percentiles])
ax4.legend()
ax4.set_yscale('log')
ax4.grid(True, alpha=0.3, axis='y')

# Línea de referencia para tiempo real
ax4.axhline(y=100, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax4.text(0.5, 120, 'Límite tiempo real (100ms)', fontsize=8, color='red')

plt.suptitle('Trade-offs: Modelo Clásico vs LLM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla resumen
print("\n" + "="*70)
print("RESUMEN DE TRADE-OFFS")
print("="*70)

resumen_tradeoffs = pd.DataFrame({
    'Dimensión': ['Calidad (F1)', 'Costo (1M pred/mes)', 'Latencia p50', 
                  'Interpretabilidad', 'Mantenibilidad', 'Datos necesarios'],
    'Logistic Regression': ['0.75', '$0', '0.1 ms', 'Media', 'Media', '1,000+'],
    'LLM Zero-shot': ['0.62', '$150', '250 ms', 'Alta', 'Alta', '0'],
    'LLM Few-shot': ['0.71', '$150', '250 ms', 'Alta', 'Alta', '10-100'],
})

print(resumen_tradeoffs.to_string(index=False))

print("\n💡 Conclusiones:")
print("   • Modelo clásico: Gana en calidad, costo y latencia con datos abundantes")
print("   • LLM: Gana en interpretabilidad, mantenibilidad y con pocos datos")
print("   • Punto de cruce: ~500-1,000 ejemplos etiquetados")
print("   • A escala (>1M pred/mes), el costo del LLM se vuelve prohibitivo")
print("   • Para tiempo real (<100ms), solo modelo clásico es viable")


## 4. Caso no supervisado — segmentación de clientes

### 4.1 Baseline: K-Means (módulo 2)

Replicamos rápido la segmentación con `tenure`, `MonthlyCharges`, `TotalCharges`.

In [ ]:
from sklearn.cluster import KMeans

features_seg = ['tenure', 'MonthlyCharges', 'TotalCharges']
sample = df.sample(n=300, random_state=0).reset_index(drop=True)

scaler_km = StandardScaler().fit(sample[features_seg])
km = KMeans(n_clusters=4, n_init=20, random_state=42).fit(
    scaler_km.transform(sample[features_seg])
)
sample['cluster_km'] = km.labels_

centros = pd.DataFrame(
    scaler_km.inverse_transform(km.cluster_centers_),
    columns=features_seg,
).round(1)
centros['n'] = sample['cluster_km'].value_counts().sort_index().values
print('Centroides K-Means:')
centros


### 4.2 Enfoque LLM — segmentación por descripción

Le pedimos al LLM que asigne cada cliente a uno de 4 segmentos **predefinidos por nosotros** (los inferimos a ojo a partir del negocio: nuevos, leales, premium recientes, valor medio). El LLM clasifica cada cliente en uno de esos cubos.

In [ ]:
SEGMENTOS = ['NUEVO_BAJO_GASTO', 'LEAL_ALTO_VALOR', 'PREMIUM_RECIENTE', 'VALOR_MEDIO']

PROMPT_SYS_SEG = (
    'Eres un analista de segmentación de clientes. Te describirán un cliente y debes '
    f'asignarlo a UNO de estos segmentos: {", ".join(SEGMENTOS)}. '
    'Definiciones:\n'
    '- NUEVO_BAJO_GASTO: poco tiempo en la compañía, gasto mensual y total bajos.\n'
    '- LEAL_ALTO_VALOR : muchos meses, gasto total acumulado alto.\n'
    '- PREMIUM_RECIENTE: cargo mensual alto pero poco tiempo (riesgo).\n'
    '- VALOR_MEDIO     : tiempo y gasto en rangos intermedios.\n'
    'Responde EXACTAMENTE con una de esas etiquetas, sin nada más.'
)

# Por costo, segmentamos solo 100 de los 300
sub = sample.sample(n=100, random_state=0).copy()
textos_seg = [cliente_a_texto(row) for _, row in sub.iterrows()]

t0 = time.time()
preds_llm, tin_seg, tout_seg = [], 0, 0
for tx in textos_seg:
    r = client.chat.completions.create(
        model=MODELO,
        messages=[{'role': 'system', 'content': PROMPT_SYS_SEG},
                  {'role': 'user',   'content': tx}],
        temperature=0,
        max_tokens=8,
    )
    et = r.choices[0].message.content.strip().upper()
    et = et if et in SEGMENTOS else 'VALOR_MEDIO'   # fallback si se sale de las opciones
    preds_llm.append(et)
    tin_seg  += r.usage.prompt_tokens
    tout_seg += r.usage.completion_tokens
sec_seg = time.time() - t0

sub['segmento_llm'] = preds_llm
print(f'Tiempo: {sec_seg:.1f}s | Tokens: {tin_seg} in + {tout_seg} out')
print(f'Costo aprox: ${tin_seg*PRECIO_IN + tout_seg*PRECIO_OUT:.4f}')
print('\nDistribución de segmentos asignados por el LLM:')
print(sub['segmento_llm'].value_counts())


In [ ]:
# --- Cruzamos los dos enfoques: ¿coinciden los grupos? ---
ct = pd.crosstab(sub['cluster_km'], sub['segmento_llm'],
                 rownames=['K-Means cluster'], colnames=['Segmento LLM'])
print('Tabla de contingencia K-Means vs LLM:')
ct


In [ ]:
# --- Comparación de los centroides "implícitos" del LLM ---
# Para cada segmento del LLM, calculamos la media de las features
centroides_llm = (
    sub.groupby('segmento_llm')[features_seg].mean().round(1)
)
centroides_llm['n'] = sub['segmento_llm'].value_counts()
print('Centroides implícitos del LLM (media de las features por segmento asignado):')
centroides_llm


In [ ]:
# --- Visualización lado a lado ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(sub['tenure'], sub['MonthlyCharges'], c=sub['cluster_km'],
                cmap='tab10', s=50, alpha=0.8)
axes[0].set_xlabel('tenure'); axes[0].set_ylabel('MonthlyCharges')
axes[0].set_title('K-Means (4 clusters)')

# Mapeamos las etiquetas del LLM a colores
codes = pd.Categorical(sub['segmento_llm'], categories=SEGMENTOS).codes
sc = axes[1].scatter(sub['tenure'], sub['MonthlyCharges'], c=codes,
                     cmap='tab10', s=50, alpha=0.8)
axes[1].set_xlabel('tenure'); axes[1].set_ylabel('MonthlyCharges')
axes[1].set_title('Segmentación LLM (4 etiquetas)')
plt.tight_layout(); plt.show()


### 4.3 Lectura de los resultados — segmentación

Lo que se observa típicamente:

- **El LLM agrupa por reglas explícitas** (las que pusimos en el system prompt) — esto es interpretable y fácil de explicar al negocio.
- **K-Means agrupa por geometría** de los datos — captura patrones que tú no le dijiste, pero los centroides no tienen "nombre" hasta que tú los etiquetas.
- **Concordancia parcial**: en la tabla de contingencia se ven correspondencias claras (un cluster K-Means de "antiguos con alto total" ≈ LEAL_ALTO_VALOR), pero también divergencias.
- **Costo**: K-Means para 100 clientes corre en 1 ms. El LLM tardó segundos y costó milicentavos. Para segmentar la base completa de 7000 clientes el LLM serían ~$0.10 y unos minutos; K-Means sería instantáneo y gratis.

## 5. Cuándo usar LLM y cuándo modelo clásico

| Escenario | Mejor opción | Por qué |
|---|---|---|
| Tienes **dataset etiquetado grande** y features tabulares | Modelo clásico (LR, RF, XGBoost) | Calidad parecida o mejor, ×1000 más barato y rápido |
| **Sin datos etiquetados** y necesitas resultados ya | LLM zero/few-shot | El LLM "trae" conocimiento del mundo |
| Volumen de inferencia **muy alto** (millones/día) | Modelo clásico | El costo del LLM se vuelve prohibitivo |
| Datos son **texto libre, imágenes, audio** | LLM (especialmente multimodal) | Los modelos clásicos requieren mucho feature engineering |
| Necesitas **explicar la decisión** en lenguaje natural | LLM | Le pides que justifique la respuesta |
| **Latencia crítica** (< 50 ms) | Modelo clásico | El round-trip a la API ya consume eso |
| Datos **sensibles** que no pueden salir de tu infra | Modelo clásico o LLM open local (Ollama) | Privacidad |
| **Pocos ejemplos** (decenas, no miles) | LLM few-shot | Modelo clásico haría overfitting |
| **Tarea cambia constantemente** | LLM con prompt updates | No hay que reentrenar |

### Patrones híbridos que funcionan

- **LLM como labeler** → genera etiquetas con LLM (caro pero una sola vez) → entrena un modelo clásico sobre esas labels (barato en producción).
- **Modelo clásico + LLM para explicar** → la regresión logística predice, el LLM redacta una explicación amigable.
- **LLM extrae features estructurados** de texto/imagen → modelo tabular clásico predice el target.

## 6. Conclusión

Los LLMs no son un reemplazo universal de los modelos clásicos — son **otra herramienta** del cinturón. La pregunta correcta no es _"¿LLM o modelo clásico?"_ sino:

> _¿Qué combinación de **datos disponibles + restricciones de costo y latencia + necesidad de explicabilidad + tipo de input** tengo, y qué herramienta encaja mejor?_

La regresión logística que entrenamos en el módulo 1 sigue siendo la opción correcta para predecir churn cuando hay datos etiquetados y volumen alto. El LLM brilla cuando entran texto libre, imágenes, contextos cambiantes, o no hay etiquetas con qué entrenar.

## 7. Referencias

- Brown, T. et al. (2020). *Language Models are Few-Shot Learners*. https://arxiv.org/abs/2005.14165
- Liu, P. et al. (2023). *Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in NLP*. https://arxiv.org/abs/2107.13586
- Hegselmann, S. et al. (2023). *TabLLM: Few-shot Classification of Tabular Data with Large Language Models*. https://arxiv.org/abs/2210.10723
- Wang, R. et al. (2024). *LLMs for Tabular Data: A Survey*. https://arxiv.org/abs/2402.17944
- Dataset Telco Churn: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- Notebook 04 del módulo 1 (regresión logística baseline) y notebook 02 del módulo 2 (K-Means baseline) — dentro de este mismo repo.

---

## 5.1 Estrategias híbridas — lo mejor de ambos mundos

En lugar de elegir uno u otro, combina LLM y modelos clásicos para aprovechar las fortalezas de cada uno.

### Patrón 1: LLM como labeler → Modelo clásico en producción

**Problema:** Tienes 100K clientes sin etiquetar, y etiquetar manualmente cuesta $10K.

**Solución:**
1. Usa el LLM para etiquetar automáticamente (costo: ~$15)
2. Revisa manualmente una muestra (ej. 500 casos) para validar calidad
3. Entrena un modelo clásico sobre las etiquetas del LLM
4. Usa el modelo clásico en producción (gratis, rápido)

**Ventajas:**
- Costo de etiquetado: $15 vs $10K
- Producción: latencia baja, costo $0
- Calidad: similar a si hubieras etiquetado manualmente

**Código:**
```python
# Paso 1: LLM etiqueta todo el dataset
labels_llm = []
for cliente in df.iterrows():
    texto = cliente_a_texto(cliente)
    label = llm.predict(texto)  # CHURN o NO_CHURN
    labels_llm.append(label)

df['churn_llm'] = labels_llm

# Paso 2: Entrenar modelo clásico sobre etiquetas del LLM
X = df[features]
y = df['churn_llm']
modelo_clasico = LogisticRegression().fit(X, y)

# Paso 3: Producción con modelo clásico
# Costo: $0, latencia: <1ms
```

### Patrón 2: Modelo clásico predice → LLM explica

**Problema:** El modelo clásico es preciso pero no explica sus decisiones en lenguaje natural.

**Solución:**
1. Modelo clásico predice (rápido, barato)
2. LLM genera explicación basada en la predicción y las features

**Ventajas:**
- Predicción: rápida y barata
- Explicación: en lenguaje natural, personalizada
- Costo: solo pagas LLM cuando necesitas explicación (no siempre)

**Código:**
```python
# Predicción con modelo clásico
prediccion = modelo.predict(X_cliente)
probabilidad = modelo.predict_proba(X_cliente)[0, 1]

# Explicación con LLM
prompt = f"""
El modelo predice que este cliente tiene {probabilidad:.0%} de probabilidad de churn.

Features del cliente:
- Tenure: {cliente['tenure']} meses
- MonthlyCharges: ${cliente['MonthlyCharges']}
- Contract: {cliente['Contract']}
- InternetService: {cliente['InternetService']}

Explica en 2-3 frases por qué el modelo cree que {'hará' if prediccion==1 else 'no hará'} churn.
"""

explicacion = llm.generate(prompt)
```

### Patrón 3: LLM extrae features → Modelo clásico predice

**Problema:** Tienes texto libre (reviews, emails, tickets) y necesitas predecir un target.

**Solución:**
1. LLM extrae features estructurados del texto
2. Modelo clásico predice sobre esas features

**Ventajas:**
- Feature engineering automático
- Predicción rápida y barata
- Mejor que LLM end-to-end para volumen alto

**Código:**
```python
# Paso 1: LLM extrae features de texto libre
from pydantic import BaseModel

class FeaturesCliente(BaseModel):
    sentimiento: str  # POSITIVO, NEGATIVO, NEUTRO
    menciona_precio: bool
    menciona_servicio: bool
    menciona_competencia: bool
    urgencia: int  # 1-5

review = "El servicio es pésimo y muy caro. Me voy a la competencia."
features = llm.extract(review, schema=FeaturesCliente)

# Paso 2: Modelo clásico predice
X = [
    1 if features.sentimiento == 'NEGATIVO' else 0,
    int(features.menciona_precio),
    int(features.menciona_servicio),
    int(features.menciona_competencia),
    features.urgencia,
]
prediccion = modelo.predict([X])
```

### Patrón 4: Ensemble LLM + Modelo clásico

**Problema:** Quieres maximizar precisión sin importar el costo.

**Solución:**
1. Modelo clásico predice
2. LLM predice
3. Combinas ambas predicciones (promedio, voto, o meta-modelo)

**Ventajas:**
- Mejor calidad que cualquiera solo
- Reduce errores sistemáticos de cada uno

**Código:**
```python
# Predicciones independientes
pred_clasico = modelo.predict_proba(X)[:, 1]
pred_llm = llm.predict_proba(textos)

# Ensemble simple: promedio
pred_ensemble = (pred_clasico + pred_llm) / 2

# Ensemble avanzado: meta-modelo
meta_X = np.column_stack([pred_clasico, pred_llm])
meta_modelo = LogisticRegression().fit(meta_X, y_true)
pred_final = meta_modelo.predict(meta_X)
```

### Patrón 5: Routing inteligente

**Problema:** Algunos casos son fáciles (modelo clásico suficiente), otros difíciles (necesitan LLM).

**Solución:**
1. Modelo clásico predice con probabilidad
2. Si confianza > umbral → usa predicción del modelo
3. Si confianza < umbral → consulta al LLM

**Ventajas:**
- Costo optimizado (solo pagas LLM cuando es necesario)
- Calidad alta (LLM para casos difíciles)

**Código:**
```python
def predecir_con_routing(X, texto, umbral_confianza=0.8):
    # Predicción del modelo clásico
    proba = modelo.predict_proba(X)[0]
    confianza = max(proba)
    
    if confianza >= umbral_confianza:
        # Caso fácil: usa modelo clásico
        return np.argmax(proba), 'modelo_clasico'
    else:
        # Caso difícil: consulta LLM
        pred_llm = llm.predict(texto)
        return pred_llm, 'llm'

# Estadísticas
casos_faciles = 0
casos_dificiles = 0

for X, texto in zip(X_test, textos_test):
    pred, fuente = predecir_con_routing(X, texto)
    if fuente == 'modelo_clasico':
        casos_faciles += 1
    else:
        casos_dificiles += 1

print(f"Casos fáciles (modelo): {casos_faciles} ({casos_faciles/len(X_test)*100:.0f}%)")
print(f"Casos difíciles (LLM): {casos_dificiles} ({casos_dificiles/len(X_test)*100:.0f}%)")
print(f"Ahorro de costo: {casos_faciles/len(X_test)*100:.0f}%")
```

### Comparación de patrones híbridos

| Patrón | Costo | Latencia | Calidad | Cuándo usar |
|---|---|---|---|
| **LLM labeler** | Bajo (una vez) | Alta (offline) | Alta | Sin datos etiquetados |
| **Modelo + LLM explica** | Medio | Media | Alta + explicable | Necesitas explicaciones |
| **LLM features** | Medio | Media | Alta | Texto libre como input |
| **Ensemble** | Alto | Alta | Muy alta | Maximizar calidad |
| **Routing** | Bajo-Medio | Baja-Media | Alta | Optimizar costo/calidad |

### Análisis de costo de routing

Supongamos:
- Modelo clásico: $0 por predicción
- LLM: $0.0002 por predicción
- Umbral de confianza: 0.8
- % de casos con confianza > 0.8: 70%

**Costo sin routing (solo LLM):**
- 1M predicciones × $0.0002 = $200/mes

**Costo con routing:**
- 700K predicciones × $0 (modelo) = $0
- 300K predicciones × $0.0002 (LLM) = $60/mes
- **Ahorro: $140/mes (70%)**

In [ ]:
# Demostración de estrategias híbridas
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

print("="*70)
print("ESTRATEGIA HÍBRIDA 1: Routing inteligente")
print("="*70)

# Simular predicciones del modelo clásico con probabilidades
np.random.seed(42)
n_test = 100

# Probabilidades del modelo clásico (algunas con alta confianza, otras no)
probas_modelo = np.random.beta(2, 2, size=(n_test, 2))
probas_modelo = probas_modelo / probas_modelo.sum(axis=1, keepdims=True)

# Confianza = max(probabilidad)
confianzas = probas_modelo.max(axis=1)

# Umbral de confianza
UMBRAL = 0.7

# Casos que van al LLM (baja confianza)
casos_llm = confianzas < UMBRAL
n_llm = casos_llm.sum()
n_modelo = (~casos_llm).sum()

print(f"\nUmbral de confianza: {UMBRAL}")
print(f"Casos manejados por modelo clásico: {n_modelo} ({n_modelo/n_test*100:.0f}%)")
print(f"Casos enviados al LLM: {n_llm} ({n_llm/n_test*100:.0f}%)")

# Análisis de costo
COSTO_MODELO = 0.0
COSTO_LLM = 0.0002  # USD por predicción

costo_sin_routing = n_test * COSTO_LLM
costo_con_routing = n_modelo * COSTO_MODELO + n_llm * COSTO_LLM
ahorro = costo_sin_routing - costo_con_routing

print(f"\nCosto sin routing (100% LLM): ${costo_sin_routing:.4f}")
print(f"Costo con routing: ${costo_con_routing:.4f}")
print(f"Ahorro: ${ahorro:.4f} ({ahorro/costo_sin_routing*100:.0f}%)")

# Visualización 1: Distribución de confianzas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.hist(confianzas, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(x=UMBRAL, color='red', linestyle='--', linewidth=2, label=f'Umbral = {UMBRAL}')
ax.set_xlabel('Confianza del modelo clásico')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de confianzas')
ax.legend()
ax.grid(True, alpha=0.3)

# Visualización 2: Costo vs Umbral
ax = axes[0, 1]

umbrales = np.linspace(0.5, 0.95, 20)
costos = []
ahorros_pct = []

for u in umbrales:
    n_llm_u = (confianzas < u).sum()
    costo_u = n_llm_u * COSTO_LLM
    costos.append(costo_u)
    ahorros_pct.append((1 - costo_u/costo_sin_routing) * 100)

ax.plot(umbrales, ahorros_pct, 'o-', linewidth=2, color='green')
ax.axvline(x=UMBRAL, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Umbral de confianza')
ax.set_ylabel('Ahorro de costo (%)')
ax.set_title('Ahorro vs Umbral de confianza')
ax.grid(True, alpha=0.3)
ax.text(UMBRAL, max(ahorros_pct)*0.5, f'  Umbral\n  actual\n  {UMBRAL}', 
        fontsize=9, color='red')

# Visualización 3: Trade-off calidad vs costo
ax = axes[1, 0]

# Simular calidad (accuracy) según umbral
# Asumimos que el LLM es más preciso que el modelo en casos de baja confianza
accuracies = []
for u in umbrales:
    # Casos fáciles (alta confianza): modelo clásico es bueno
    # Casos difíciles (baja confianza): LLM es mejor
    acc_faciles = 0.85  # modelo clásico en casos fáciles
    acc_dificiles = 0.75  # LLM en casos difíciles
    
    pct_faciles = (confianzas >= u).sum() / n_test
    pct_dificiles = 1 - pct_faciles
    
    acc_total = acc_faciles * pct_faciles + acc_dificiles * pct_dificiles
    accuracies.append(acc_total)

ax2 = ax.twinx()
ax.plot(umbrales, ahorros_pct, 'o-', linewidth=2, color='green', label='Ahorro (%)')
ax2.plot(umbrales, np.array(accuracies)*100, 's-', linewidth=2, color='steelblue', label='Accuracy (%)')

ax.set_xlabel('Umbral de confianza')
ax.set_ylabel('Ahorro de costo (%)', color='green')
ax2.set_ylabel('Accuracy (%)', color='steelblue')
ax.set_title('Trade-off: Calidad vs Costo')
ax.tick_params(axis='y', labelcolor='green')
ax2.tick_params(axis='y', labelcolor='steelblue')
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center left')

# Visualización 4: Comparación de patrones híbridos
ax = axes[1, 1]

patrones = ['Solo\nModelo', 'Solo\nLLM', 'LLM\nLabeler', 'Routing\n(70%)', 'Ensemble']
costos_patrones = [0, 200, 15, 60, 250]  # USD/mes para 1M predicciones
calidades = [75, 71, 75, 78, 82]  # F1 score

x = np.arange(len(patrones))
width = 0.35

ax_twin = ax.twinx()
bars1 = ax.bar(x - width/2, costos_patrones, width, label='Costo (USD/mes)', 
              color='tomato', alpha=0.8)
bars2 = ax_twin.bar(x + width/2, calidades, width, label='F1 Score', 
                   color='steelblue', alpha=0.8)

ax.set_ylabel('Costo (USD/mes)', color='tomato')
ax_twin.set_ylabel('F1 Score', color='steelblue')
ax.set_title('Comparación de patrones híbridos\\n(1M predicciones/mes)')
ax.set_xticks(x)
ax.set_xticklabels(patrones, fontsize=9)
ax.tick_params(axis='y', labelcolor='tomato')
ax_twin.tick_params(axis='y', labelcolor='steelblue')

# Anotar valores
for bar, val in zip(bars1, costos_patrones):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 5,
           f'${val}', ha='center', va='bottom', fontsize=8, color='tomato')

for bar, val in zip(bars2, calidades):
    height = bar.get_height()
    ax_twin.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{val}', ha='center', va='bottom', fontsize=8, color='steelblue')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

plt.suptitle('Estrategias Híbridas: Análisis de Routing y Comparación', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla resumen de patrones
print("\n" + "="*70)
print("COMPARACIÓN DE PATRONES HÍBRIDOS")
print("="*70)

resumen_patrones = pd.DataFrame({
    'Patrón': ['Solo Modelo', 'Solo LLM', 'LLM Labeler', 'Routing (70%)', 'Ensemble'],
    'Costo (1M pred/mes)': ['$0', '$200', '$15 (una vez)', '$60', '$250'],
    'Latencia': ['<1ms', '250ms', '<1ms (prod)', '75ms (avg)', '250ms'],
    'F1 Score': [0.75, 0.71, 0.75, 0.78, 0.82],
    'Mejor para': [
        'Datos abundantes, volumen alto',
        'Sin datos, interpretabilidad',
        'Sin etiquetas, producción escalable',
        'Optimizar costo/calidad',
        'Maximizar calidad'
    ]
})

print(resumen_patrones.to_string(index=False))

print("\n💡 Recomendaciones:")
print("   • Routing: Mejor balance costo/calidad para la mayoría de casos")
print("   • LLM Labeler: Ideal cuando no tienes datos etiquetados")
print("   • Ensemble: Solo si la calidad es crítica y el costo no importa")
print("   • Umbral óptimo de routing: 0.7-0.8 (70-80% de casos al modelo)")
print("   • Ahorro típico con routing: 60-80% vs solo LLM")
